# BPE Tokenizer Walkthrough

A quick, interactive look at the byte-level BPE tokenizer in `src/tokenizer/bpe.py`:
training it on a small corpus, inspecting the merges it learns, and
verifying encode/decode invertibility -- including on text the tokenizer
never saw during training.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.tokenizer.bpe import BPETokenizer

with open('../data/raw/corpus.txt') as f:
    text = f.read()

print(f'corpus length: {len(text):,} characters')

In [ ]:
tok = BPETokenizer.train(text, vocab_size=512, verbose=True)
print(f'\nfinal vocab size: {tok.vocab_size}')
print(f'number of learned merges: {len(tok.merges)}')

## Inspect a few learned merges

Early merges tend to be very common short sequences (like `t` + `h` -> `th`);
later merges combine those into whole common words.

In [ ]:
for i, pair in enumerate(tok.merges[:10]):
    new_id = 256 + i
    print(f'merge {i:3d}: {pair} -> id {new_id} = {tok.vocab[new_id]!r}')

## Encode / decode round trip

Try some text the tokenizer definitely never saw during training,
including a non-Latin script and emoji -- byte-level BPE handles all of it
because every possible byte is representable.

In [ ]:
samples = [
    'the quick brown fox',
    'a completely unseen sentence about spaceships and lasers',
    'Héllo wörld with accents',
    '你好，世界！',
    'emoji test: 🚀🔥😀',
]

for s in samples:
    ids = tok.encode(s)
    back = tok.decode(ids)
    print(f'{s!r:55s} -> {len(ids):3d} tokens -> round_trip_ok={back == s}')

In [ ]:
tok.save('../src/tokenizer/vocab.json')
print('saved tokenizer to ../src/tokenizer/vocab.json')